In [1]:
import os 
import nest_asyncio

nest_asyncio.apply()

API_KEY = os.environ['API_KEY']
SECRET_KEY = os.environ['SECRET_KEY']

In [2]:
from alpaca.trading.client import TradingClient

trading_client = TradingClient(API_KEY,SECRET_KEY, paper=True)

SYMBOL = 'TSLA'
free_cash_perc = 0.1
OVERBOUGHT_THRESH = 70
OVERSOLD_THRESH = 30
CLOSE_POSITION_THRESH = 50

In [1]:
from alpaca.trading.requests import MarketOrderRequest

def place_market_order(side, val=None):
    if side == 'buy':
        
        order_data = MarketOrderRequest(
              symbol=symbol,
              notional=val,
              side=OrderSide.BUY,
                timeinforce='day')
        
        return trading_client.submit_order(order_data=order_data)
        
    elif side=='sell':
        
        return trading_client.close_position(SYMBOL)

In [2]:
from alpaca.trading.enums import OrderSide, TimeInForce
from pandas_ta.momentum import rsi
import pandas as pd

bars = []
has_position = False
is_long  = False
is_short = False
async def on_new_bar(bar):
   global bars
   global has_position
   bars.append({k: v for k, v in bar})
   print("New bar!")
   print(f"Bar # {len(bars)}")

   if len(bars)>=15: # Need at least 14 bars to calculate RSI_14
     # Get as Bars DataFrame
     df = pd.DataFrame(bars)
     df.set_index('timestamp',inplace=True)
     # Calculate Latest RSI
     rsi_value = rsi(df.close.iloc[-15:])[-1]
     print(f"RSI IS {rsi_value}")
     # Trading Logic
     if rsi_value<=OVERSOLD_THRESH and not has_position:
       val = float(trading_client.get_account().buying_power)*free_cash_perc
       place_market_order('buy', val)
       has_position = True
       is_long = True
     elif rsi_value>=OVERSOLD_THRESH and not has_position:
       place_market_order('sell')
       has_position = True
       is_long = False
     elif rsi_value>=CLOSE_POSITION_THRESH and is_long:
       has_position = False
       is_long = False
       place_market_order('sell')
     elif rsi_value<=CLOSE_POSITION_THRESH and is_short:
       val = float(trading_client.get_account().buying_power)*free_cash_perc
       has_position = False
       is_short = False
       place_market_order('buy', val)

In [3]:
from alpaca.data.live.stock import StockDataStream
stock_stream = StockDataStream(API_KEY,SECRET_KEY)

NameError: name 'API_KEY' is not defined

In [ ]:
stock_stream.subscribe_bars(on_new_bar,SYMBOL)

In [ ]:
stock_stream.run()